In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient
import logging
import warnings

from agoradatatools.gx import GreatExpectationsRunner
from synapseclient.models import File

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members import ExpectColumnValuesToHaveListMembers
from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange
from expectations.expect_column_values_to_have_unique_list_members import ExpectColumnValuesToHaveUniqueListMembers

# Create Expectation Suite for Marmoset Model Overview Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# preprod
marmo_overview_file = await File(id="syn76226134").get_async()

## Create Validator Object on Data File

In [ ]:
# dtype=False matches GreatExpectationsRunner.run(), which does not infer dtypes.
df = pd.read_json(marmo_overview_file.path, dtype=False)
nested_columns = ["biomarkers", "study_data"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "marmo_overview"

## Add Expectations to Validator Object For Each Column

In [ ]:
# Fails on a new/unknown column, or a missing expected one.
validator.expect_table_columns_to_match_set(
    column_set=["name", "model_type", "biomarkers", "study_data", "modified_genes", "available_data"],
    exact_match=True,
)

In [ ]:
# name
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")
validator.expect_column_values_to_be_unique("name")

In [ ]:
# model_type (be_in_set values aligned with ui_config model_type_filter_vals)
validator.expect_column_values_to_be_of_type("model_type", "str")
validator.expect_column_values_to_not_be_null("model_type")
validator.expect_column_values_to_be_in_set("model_type", ["Familial AD", "Late Onset AD"])

In [ ]:
# modified_genes (list_members aligned with ui_config marmo_mo_modified_gene_filter_vals)
validator.expect_column_values_to_be_of_type("modified_genes", "list")
validator.expect_column_values_to_not_be_null("modified_genes")
validator.expect_column_values_to_have_list_length_in_range(column="modified_genes", list_length_range=[1, 1])
validator.expect_column_values_to_have_list_members_of_type(column="modified_genes", member_type="str")
validator.expect_column_values_to_have_unique_list_members(column="modified_genes")
validator.expect_column_values_to_have_list_members(column="modified_genes", list_members={"PSEN1"})

In [ ]:
# available_data (list_members aligned with the Available Data values in ui_config marmo_mo_filter_values)
validator.expect_column_values_to_be_of_type("available_data", "list")
validator.expect_column_values_to_not_be_null("available_data")
validator.expect_column_values_to_have_list_length_in_range(column="available_data", list_length_range=[1, 1])
validator.expect_column_values_to_have_list_members_of_type(column="available_data", member_type="str")
validator.expect_column_values_to_have_unique_list_members(column="available_data")
validator.expect_column_values_to_have_list_members(column="available_data", list_members={"Plasma Biomarkers"})

In [ ]:
# biomarkers (nullable link column)
# The schema validates the nested link_url against the marmoset biomarkers page URL pattern.
with open("../src/agoradatatools/great_expectations/gx/json_schemas/marmo_overview/biomarkers_schema.json", "r") as file:
    biomarkers_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("biomarkers", json_schema=biomarkers_schema)

In [ ]:
# study_data (required link column - must never be null)
# The schema pins link_url to the marmoset study (syn61849889), which ui_config also hard-codes.
with open("../src/agoradatatools/great_expectations/gx/json_schemas/marmo_overview/study_data_schema.json", "r") as file:
    study_data_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("study_data", json_schema=study_data_schema)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()